# Vector Quantization and the Lloyd–Max Optimality Conditions — companion notebook

This is the **narrative** half of the Python pillar. The reusable, tested implementation
lives next door in [`vector_quantization_lloyd_max.py`](vector_quantization_lloyd_max.py);
here we import it and walk the topic section by section, so every claim the page makes
renders as executed output.

**Two artifacts, one source of truth.** `vector_quantization_lloyd_max.py` owns the
numbers — its `test_*` assertions encode the theorems, and `viz_constants()` prints what
`VectorQuantizationLaboratory.tsx` mirrors to the decimal. This notebook never redefines
the math; it calls in.

In [ ]:
import sys
import pathlib

# Make the module importable whether the kernel starts in the repo root or here.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "vector-quantization-lloyd-max",
              pathlib.Path("notebooks/vector-quantization-lloyd-max")):
    if (_cand / "vector_quantization_lloyd_max.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
import vector_quantization_lloyd_max as vq

## 1. The quantization problem and Lloyd's descent

A quantizer factors as a decoder composed with an encoder, $Q=\beta\circ\alpha$, and its
quality is the distortion $D=\mathbb{E}\lVert X-Q(X)\rVert^2$. We run Lloyd's algorithm on
a small two-dimensional cloud (the one the laboratory draws) and watch the distortion fall
monotonically to a fixed point — **Theorem 3**.

In [ ]:
X, labels = vq.toy_cloud_2d()
C, lab, hist = vq.lloyd(X, 3, vq.SEED_DEFAULT, tol=1e-12)
print(f"toy cloud: {X.shape[0]} points in [0,10]^2, k = 3 codewords")
print(f"distortion history : {[round(h, 2) for h in hist]}")
print(f"converged distortion {hist[-1]:.2f} in {len(hist)} iterations")
vq.test_distortion_monotone()

## 2. The two optimality conditions at the fixed point

At convergence Lloyd satisfies both conditions at once: every point sits in the cell of its
**nearest** codeword (the encoder is optimal, **Theorem 1**) and every codeword is the
**conditional mean** of its cell (the decoder is optimal, **Theorem 2**). One more
assignment changes no labels and one more update moves no codeword.

In [ ]:
vq.test_nearest_neighbor_condition()
vq.test_centroid_condition()
vq.test_convergence_fixed_point()

## 3. $k$-means is Lloyd under the empirical measure

Replace the source distribution by the empirical measure over a finite sample and the
distortion becomes the within-cluster sum of squares, the centroid becomes the sample mean,
and Lloyd becomes $k$-means (**Proposition 1**). From the same initialization our
implementation matches `scipy.cluster.vq.kmeans2`.

In [ ]:
vq.test_kmeans_objective_equals_sse()
vq.validate_against_kmeans2()

## 4. A fixed point is only a local optimum

The distortion is non-convex in the codebook, so the optimum Lloyd reaches depends on the
initialization. Across many random seeds the final distortion has real spread, and
$k$-means++ lowers the mean. The three seeds the laboratory's **Reseed** button cycles
through reach three different optima on the *same* cloud.

In [ ]:
vq.test_local_optima_sensitivity()
vq.test_toy_local_optima()
print()
print("Seeds the laboratory bakes (mirrored to the decimal in the .tsx):")
vq.viz_constants()

## 5. The empty-cell guard

On a finite sample a codeword can capture no points, making its conditional mean a $0/0$. We
reseed an orphaned codeword onto the worst-served point — a deterministic furthest-point
repair that stays finite and never raises distortion.

In [ ]:
vq.test_empty_cluster_guard()

## 6. Zador's law: distortion scales as $k^{-2/d}$

The optimal-quantizer distortion falls as $k^{-2/d}$ — the curse of dimensionality in
quantization form. We verify the **exponent** (not Zador's constant) on uniform sources,
where the high-rate regime is reachable: the log-log slope is near $-2/d$.

In [ ]:
s1 = vq.zador_slope(1, ks=(8, 16, 32, 64, 128, 256))
s2 = vq.zador_slope(2, ks=(8, 16, 32, 64, 128, 256))
print(f"uniform-source slope:  d=1 -> {s1:.3f} (Zador -2),  d=2 -> {s2:.3f} (Zador -1)")
vq.test_zador_scaling()

## 7. Finance case study: the rate–distortion tradeoff

Train codebooks of growing size on a synthetic 256-d finance cloud and read off the
rate ($\log_2 k$ bits per vector) against distortion and retained recall@10. Distortion
falls monotonically but slowly — the high effective $d$ in $k^{-2/d}$ — which is the wall
product quantization is built to break.

In [ ]:
vq.finance_demo()
vq.test_rate_distortion_monotone()

## All claims verified

Every theorem on the page is an executed assertion above. The toy cloud, its seed
codebooks, and the converged distortions are the exact numbers
`VectorQuantizationLaboratory.tsx` mirrors; the finance rate–distortion grid is the
second laboratory panel. Re-run top to bottom to reproduce the topic end to end.